<a href="https://colab.research.google.com/github/smosharof/Resume.Walkthrough/blob/main/BERT_Compare_Documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install PyPDF2 transformers torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [2]:
import PyPDF2
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file."""
    text = ""
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text

def chunk_text(text, chunk_size=512, chunk_overlap=100):
    """Splits text into smaller chunks."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - chunk_overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

def get_bert_embeddings(text_list, model, tokenizer):
    """Gets BERT embeddings for a list of texts."""

    encoded_input = tokenizer.batch_encode_plus(
        text_list,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )
    input_ids = encoded_input['input_ids']
    attention_mask = encoded_input['attention_mask']

    with torch.no_grad():
        model_output = model(input_ids, attention_mask=attention_mask)
    # Use the CLS token embeddings (the first token)
    embeddings = model_output.last_hidden_state[:, 0, :]  # Shape: [batch_size, 768]
    return embeddings

# --- Main ---

# 1. Extract text from the PDFs
apple_text = extract_text_from_pdf("Apple 10K Q4 2024.pdf")
meta_text = extract_text_from_pdf("Meta 10K Q4 2024.pdf")

# 2. Chunk the text
apple_chunks = chunk_text(apple_text)
meta_chunks = chunk_text(meta_text)

# 3. Initialize BERT model and tokenizer
model_name = 'bert-base-uncased'  # You can experiment with other models
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

# 4. Get embeddings
apple_embeddings = get_bert_embeddings(apple_chunks, model, tokenizer)
meta_embeddings = get_bert_embeddings(meta_chunks, model, tokenizer)

# 5. Calculate similarity (compare each chunk from Apple to each chunk from Meta)
similarity_matrix = cosine_similarity(apple_embeddings, meta_embeddings)

# 6. Analyze the similarity matrix
# For example, find the most similar chunks:
import numpy as np

max_similarity = np.max(similarity_matrix)
max_row_idx, max_col_idx = np.unravel_index(np.argmax(similarity_matrix), similarity_matrix.shape)

print(f"Maximum similarity: {max_similarity:.4f}")
print(f"Apple chunk index: {max_row_idx}, Meta chunk index: {max_col_idx}")
print("\nApple Chunk:\n", apple_chunks[max_row_idx])
print("\nMeta Chunk:\n", meta_chunks[max_col_idx])

# Further analysis: You could calculate average similarity, find chunks above a threshold, etc.

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Maximum similarity: 0.9401
Apple chunk index: 46, Meta chunk index: 5

Apple Chunk:
 Financial Statements. Apple Inc. | 2024 Form 10-K | 32Apple Inc. CONSOLIDATED STATEMENTS OF CASH FLOWS (In millions) Years ended September 28, 2024September 30, 2023September 24, 2022 Cash, cash equivalents, and restricted cash and cash equivalents, beginning balances $ 30,737 $ 24,977 $ 35,929 Operating activities: Net income 93,736 96,995 99,803 Adjustments to reconcile net income to cash generated by operating activities: Depreciation and amortization 11,445 11,519 11,104 Share-based compensation expense 11,688 10,833 9,038 Other (2,266) (2,227) 1,006 Changes in operating assets and liabilities: Accounts receivable, net (3,788) (1,688) (1,823) Vendor non-trade receivables (1,356) 1,271 (7,520) Inventories (1,046) (1,618) 1,484 Other current and non-current assets (11,731) (5,684) (6,499) Accounts payable 6,020 (1,889) 9,448 Other current and non-current liabilities 15,552 3,031 6,110 Cash generated 